In [1]:
import os
import requests
import pandas as pd

PROJECT_ROOT = r"C:\Users\patel\Desktop\group_project\cardiotox-fusion"
DICTRANK_URL = "https://www.fda.gov/media/178811/download?attachment"
DICTRANK_RAW_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "dictrank_dataset_508.xlsx")

def fetch_dictrank(force_redownload=False):
    """Download DICTrank from FDA if not already present locally."""
    if os.path.exists(DICTRANK_RAW_PATH) and not force_redownload:
        print(f"DICTrank file already exists at {DICTRANK_RAW_PATH}, skipping download.")
        return

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    resp = requests.get(DICTRANK_URL, headers=headers, timeout=30)
    print(f"DICTrank download status: {resp.status_code}, content-type: {resp.headers.get('Content-Type')}")
    with open(DICTRANK_RAW_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Saved to {DICTRANK_RAW_PATH} ({os.path.getsize(DICTRANK_RAW_PATH)} bytes)")

print("Function defined.")

Function defined.


In [2]:
fetch_dictrank()

DICTrank file already exists at C:\Users\patel\Desktop\group_project\cardiotox-fusion\data\raw\dictrank_dataset_508.xlsx, skipping download.


In [3]:
def load_and_clean_dictrank():
    """Load DICTrank, clean column names, normalize concern labels, binarize."""
    df = pd.read_excel(DICTRANK_RAW_PATH)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={"DICT _ Concern": "DICT_Concern"})
    df["DICT_Concern"] = df["DICT_Concern"].str.strip().str.lower()

    print(f"Loaded {len(df)} total DICTrank rows")
    print("Concern level counts:")
    print(df["DICT_Concern"].value_counts(dropna=False))

    df_labeled = df[df["DICT_Concern"] != "ambiguous"].copy()
    df_labeled["cardiotox_label"] = df_labeled["DICT_Concern"].map({
        "no": 0, "less": 1, "most": 1
    })

    n_unmapped = df_labeled["cardiotox_label"].isna().sum()
    if n_unmapped > 0:
        print(f"WARNING: {n_unmapped} rows failed to map to a binary label -- investigate before proceeding")
    else:
        print(f"All {len(df_labeled)} labeled rows mapped cleanly to binary labels")

    return df_labeled

print("Function defined.")

Function defined.


In [4]:
df_labeled = load_and_clean_dictrank()
df_labeled.head()

Loaded 1318 total DICTrank rows
Concern level counts:
DICT_Concern
less         527
no           343
most         341
ambiguous    107
Name: count, dtype: int64
All 1211 labeled rows mapped cleanly to binary labels


,Trade Name,Generic/Proper Name(s),Active Ingredient(s),Cardiotoxicity,Label Section,DICT_Concern,Keywords,DIC Severity Level,cardiotox_label
0,Hyaluronic Acid,HYALURONIC ACID,HYALURONIC ACID,NaN,no,no,NaN,NaN,0
1,ZYPREXA Relprevv,OLANZAPINE PAMOATE,Olanzapine pamoate,Arrhythmia,wp,less,"tachycardia, bradycardia",mild,1
2,Tigan,TRIMETHOBENZAMIDE HYDROCHLORIDE,Trimethobenzamide Hydrochloride,NaN,no,no,NaN,NaN,0
3,BICILLIN L-A,PENICILLIN G BENZATHINE,PENICILLIN G BENZATHINE,Arrhythmia,ar,less,Cardiac arrest,severe,1
4,INVEGA HAFYERA,PALIPERIDONE PALMITATE,paliperidone palmitate,Arrhythmia,wp,most,QT Prolongation,moderate,1


In [5]:
CREDIBLEMEDS_PATH = os.path.join(PROJECT_ROOT, "data", "raw", "crediblemeds_qtdrugs.csv")

def cross_reference_crediblemeds(df_labeled):
    """
    Optional cross-check against a manually-downloaded CredibleMeds QTdrugs list.
    CredibleMeds requires a free registered account and its list cannot be
    scraped automatically -- this was confirmed during the project's mandatory
    trace-through phase. If the file isn't present, this step is skipped
    with a clear message, not a silent failure.
    """
    if not os.path.exists(CREDIBLEMEDS_PATH):
        print(f"\nCredibleMeds file not found at {CREDIBLEMEDS_PATH}")
        print("  -> Skipping cross-reference. To enable: register a free account at")
        print("     crediblemeds.org, manually download the QTdrugs list, and save it")
        print("     to that path. Remember to record the access date for citation.")
        return df_labeled

    cm_df = pd.read_csv(CREDIBLEMEDS_PATH)
    print(f"\nLoaded CredibleMeds list: {len(cm_df)} rows")
    print("NOTE: exact column matching logic to be added once we see the real file structure.")
    return df_labeled

print("Function defined.")

Function defined.


In [6]:
df_labeled = cross_reference_crediblemeds(df_labeled)


CredibleMeds file not found at C:\Users\patel\Desktop\group_project\cardiotox-fusion\data\raw\crediblemeds_qtdrugs.csv
  -> Skipping cross-reference. To enable: register a free account at
     crediblemeds.org, manually download the QTdrugs list, and save it
     to that path. Remember to record the access date for citation.


In [8]:
CredibleMeds file not found at C:\Users\patel\Desktop\group_project\cardiotox-fusion\data\raw\crediblemeds_qtdrugs.csv
  -> Skipping cross-reference. To enable: register a free account at
     crediblemeds.org, manually download the QTdrugs list, and save it
     to that path. Remember to record the access date for citation.

SyntaxError: invalid syntax (3476298274.py, line 1)